In [11]:
import pandas as pd
import numpy as np
import heapq
from collections import deque
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 1. SLIDING WINDOW STATISTICS CLASS
# ------------------------------------------------------------

class SlidingWindowStats:
    """Class for calculating sliding window statistics using deque"""
    def __init__(self, window_size=100):
        self.window_size = window_size
        self.window = deque(maxlen=window_size)
        self.min_heap = []  # Store larger half
        self.max_heap = []  # Store smaller half (negated)
        self.to_remove = {}
        
    def _clean_heap(self, heap, is_max_heap=False):
        while heap and self._get_top(heap, is_max_heap) in self.to_remove:
            val = self._get_top(heap, is_max_heap)
            self.to_remove[val] -= 1
            if self.to_remove[val] == 0:
                del self.to_remove[val]
            heapq.heappop(heap)
    
    def _get_top(self, heap, is_max_heap=False):
        return -heap[0] if is_max_heap else heap[0]
    
    def _balance_heaps(self):
        if len(self.max_heap) > len(self.min_heap) + 1:
            moved = -heapq.heappop(self.max_heap)
            heapq.heappush(self.min_heap, moved)
        elif len(self.min_heap) > len(self.max_heap):
            moved = heapq.heappop(self.min_heap)
            heapq.heappush(self.max_heap, -moved)
    
    def add(self, x):
        if len(self.window) == self.window_size:
            oldest = self.window[0]
            self.to_remove[oldest] = self.to_remove.get(oldest, 0) + 1
        
        self.window.append(x)
        
        if not self.max_heap or x <= -self.max_heap[0]:
            heapq.heappush(self.max_heap, -x)
        else:
            heapq.heappush(self.min_heap, x)
        
        self._clean_heap(self.max_heap, is_max_heap=True)
        self._clean_heap(self.min_heap, is_max_heap=False)
        self._balance_heaps()
    
    def get_mean(self):
        return sum(self.window) / len(self.window) if self.window else 0.0
    
    def get_median(self):
        if not self.max_heap:
            return 0.0
        self._clean_heap(self.max_heap, True)
        self._clean_heap(self.min_heap, False)
        self._balance_heaps()
        total_len = len(self.max_heap) + len(self.min_heap)
        if total_len == 0:
            return 0.0
        if total_len % 2 == 1:
            return -self.max_heap[0]
        return (-self.max_heap[0] + self.min_heap[0]) / 2.0
    
    def get_std(self):
        if len(self.window) < 2:
            return 0.0
        mean = self.get_mean()
        variance = sum((x - mean) ** 2 for x in self.window) / (len(self.window) - 1)
        return np.sqrt(variance)
    
    def get_min(self):
        return min(self.window) if self.window else 0.0
    
    def get_max(self):
        return max(self.window) if self.window else 0.0
    
    def get_range(self):
        return self.get_max() - self.get_min()
    
    def get_all_stats(self):
        return {
            'mean': self.get_mean(), 'median': self.get_median(), 'std': self.get_std(),
            'min': self.get_min(), 'max': self.get_max(), 'range': self.get_range(),
            'count': len(self.window)
        }

# ------------------------------------------------------------
# 2. SLIDING WINDOW STATISTICS FOR FIXED FEATURES
# ------------------------------------------------------------

def compute_sliding_window_stats(df, fixed_features, window_size, stats):
    """Compute sliding window statistics for fixed features"""
    available_features = [f for f in fixed_features if f in df.columns]
    if not available_features:
        raise ValueError(f"None of the fixed features found: {fixed_features}")
    
    print(f"Fixed features: {available_features}, Window: {window_size}, Stats: {stats}")
    
    n_samples, n_features, n_stats = len(df), len(available_features), len(stats)
    result = np.zeros((n_samples, n_features * n_stats))
    states = [SlidingWindowStats(window_size) for _ in range(n_features)]
    
    for i in range(n_samples):
        for f_idx, feature in enumerate(available_features):
            state = states[f_idx]
            state.add(df.iloc[i][feature])
            all_stats = state.get_all_stats()
            
            stat_values = [all_stats[s] for s in stats]
            col_idx = f_idx * n_stats
            for s_idx, val in enumerate(stat_values):
                result[i, col_idx + s_idx] = val
    
    columns = [f'{feature}_{s}' for feature in available_features for s in stats]
    return pd.DataFrame(result, columns=columns, index=df.index), available_features

# ------------------------------------------------------------
# 3. LEAKAGE-SAFE ROUTING
# ------------------------------------------------------------
# Training and inference both use the fitted K-Means predict rule. Labels never
# participate in routing or in the congestion-level assignment.

# ------------------------------------------------------------
# 4. MAIN LEAKAGE-SAFE PIPELINE
# ------------------------------------------------------------

def universal_clustering_pipeline(df, target_column='label', n_clusters=3, test_size=0.2,
                                 random_state=42, external_test_df=None, preserve_order=True,
                                 window_size=100,
                                 clustering_stats=['mean', 'median', 'std', 'min', 'max']):
    """Train and evaluate CDR-MLC without target leakage."""
    
    print("="*80)
    print("CDR-MLC PIPELINE (leakage-safe hard routing)")
    print("="*80)
    print(f"Stats: {clustering_stats}, Window: {window_size}, Features: SynAck, AckDat, TcpRtt")
    
    # Data preparation
    df_processed = df.copy()
    if target_column not in df_processed.columns:
        target_column = 'label' if 'label' in df_processed.columns else target_column
    
    # ========== LABEL ENCODING ==========
    print(f"\n🔵 Encoding labels: {df_processed[target_column].unique()}")
    label_encoder = LabelEncoder()
    encoded_labels = label_encoder.fit_transform(df_processed[target_column])
    
    # Store mapping for later use
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
    inverse_mapping = {v: k for k, v in label_mapping.items()}
    
    print(f"Label mapping: {label_mapping}")
    print(f"Classes: {len(label_encoder.classes_)}")
    
    # Keep the encoded target only in the target column. Do not create a
    # label_encoded predictor column.
    df_processed[target_column] = encoded_labels
    
    print(f"\nData: {len(df_processed)} samples, Classes: {df_processed[target_column].nunique()}")
    for label, count in df_processed[target_column].value_counts().sort_index().items():
        original_name = inverse_mapping[label]
        print(f"  Class {label} ({original_name}): {count} ({100*count/len(df_processed):.1f}%)")
    
    # Train-test split
    if external_test_df is not None:
        # Apply same encoding to test data
        test_df_raw = external_test_df.copy()
        test_df_raw[target_column] = label_encoder.transform(test_df_raw[target_column])
        
        train_df = df_processed.copy()
        test_df = test_df_raw.copy()
    elif preserve_order:
        train_dfs, test_dfs = [], []
        for label in df_processed[target_column].unique():
            class_data = df_processed[df_processed[target_column] == label]
            split_idx = int(len(class_data) * (1 - test_size))
            train_dfs.append(class_data.iloc[:split_idx])
            test_dfs.append(class_data.iloc[split_idx:])
        train_df = pd.concat(train_dfs).sort_index()
        test_df = pd.concat(test_dfs).sort_index()
    else:
        from sklearn.model_selection import train_test_split
        train_df, test_df = train_test_split(df_processed, test_size=test_size, 
                                            stratify=df_processed[target_column], random_state=random_state)
    
    print(f"\nTrain: {len(train_df)}, Test: {len(test_df)}")
    
    # Compute sliding window statistics
    fixed_features = ['SynAck', 'AckDat', 'TcpRtt']
    train_stats, used_features = compute_sliding_window_stats(train_df, fixed_features, window_size, clustering_stats)
    test_stats, _ = compute_sliding_window_stats(test_df, fixed_features, window_size, clustering_stats)
    
    train_df = pd.concat([train_df, train_stats], axis=1)
    test_df = pd.concat([test_df, test_stats], axis=1)
    
    # Clustering
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_stats)
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=random_state)
    kmeans.fit(X_train)
    
    # The same label-free routing rule is used in training and inference.
    train_df['cluster'] = kmeans.predict(X_train)
    
    X_test = scaler.transform(test_stats)
    test_df['cluster'] = kmeans.predict(X_test)
    
    # Remove clustering features
    train_df = train_df.drop(columns=train_stats.columns)
    test_df = test_df.drop(columns=test_stats.columns)
    
    # Classification features: explicitly reject the target and all target-derived
    # bookkeeping columns. Congestion features are used only by the router.
    def is_target_metadata(column):
        name = column.strip().lower().replace('-', '_').replace(' ', '_')
        return ('label' in name or name in {'class', 'target', 'y'} or
                name.startswith(('class_', 'target_', 'index_in_', 'unnamed:')))

    classification_features = [
        c for c in train_df.select_dtypes(include=[np.number]).columns
        if c not in [target_column, 'cluster'] + used_features
        and not is_target_metadata(c)
    ]
    assert target_column not in classification_features
    assert not any(is_target_metadata(c) for c in classification_features)
    print(f"\nClassification features: {len(classification_features)} (excluding {used_features})")
    print(f"Leakage check passed: target-derived columns are excluded")
    
    # Train per-cluster classifiers
    classifiers = {}
    for cid in range(n_clusters):
        train_c = train_df[train_df['cluster'] == cid]
        if len(train_c) < 5:
            print(f"Cluster {cid}: Skipped (only {len(train_c)} samples)")
            continue
        
        X_c, y_c = train_c[classification_features], train_c[target_column]
        clf = RandomForestClassifier(n_estimators=80, random_state=random_state, 
                                    class_weight='balanced', n_jobs=-1)
        clf.fit(X_c, y_c)
        classifiers[cid] = clf
        print(f"Cluster {cid}: Trained on {len(train_c)} samples, {y_c.nunique()} classes")
    
    # Leakage-safe evaluation: each sample is evaluated only by the expert
    # selected by its predicted congestion cluster.
    print("\nEvaluating test data (assigned cluster only)...")

    X_test_all = test_df[classification_features].values
    y_test_all = test_df[target_column].values
    test_clusters = test_df['cluster'].values
    predictions = np.zeros(len(X_test_all), dtype=int)

    for cid, clf in classifiers.items():
        mask = (test_clusters == cid)
        if mask.any():
            predictions[mask] = clf.predict(X_test_all[mask])

    missing_mask = ~np.isin(test_clusters, list(classifiers.keys()))
    if missing_mask.any():
        majority_class = int(train_df[target_column].mode()[0])
        predictions[missing_mask] = majority_class
        print(f"Warning: {missing_mask.sum()} samples routed to an untrained cluster; "
              f"using majority class {majority_class}")

    # Convert predictions back to original labels for display
    predictions_original = [inverse_mapping[p] for p in predictions]
    y_test_original = [inverse_mapping[y] for y in y_test_all]
    
    # Results with original labels
    results = {
        'accuracy': accuracy_score(y_test_original, predictions_original),
        'f1_weighted': f1_score(y_test_original, predictions_original, average='weighted'),
        'f1_macro': f1_score(y_test_original, predictions_original, average='macro'),
        'recall_weighted': recall_score(y_test_original, predictions_original, average='weighted'),
        'precision_weighted': precision_score(y_test_original, predictions_original, average='weighted'),
        'n_samples': len(predictions),
        'confusion_matrix': confusion_matrix(y_test_original, predictions_original),
        'classification_report': classification_report(y_test_original, predictions_original, digits=4),
        'label_encoder': label_encoder,
        'label_mapping': label_mapping,
        'inverse_mapping': inverse_mapping
    }
    
    print(f"\n{'='*60}")
    print("TEST RESULTS (leakage-safe assigned cluster)")
    print(f"{'='*60}")
    print(f"Accuracy: {results['accuracy']:.4f}")
    print(f"F1 Weighted: {results['f1_weighted']:.4f}")
    print(f"Recall Weighted: {results['recall_weighted']:.4f}")
    print(f"Precision Weighted: {results['precision_weighted']:.4f}")
    
    return {
        'train_df': train_df, 'test_df': test_df, 'classifiers': classifiers,
        'test_results': results, 'used_fixed_features': used_features,
        'classification_features': classification_features, 'n_classes': len(np.unique(y_test_all)),
        'kmeans_model': kmeans, 'scaler': scaler, 'target_column': target_column,
        'window_size': window_size, 'clustering_stats': clustering_stats,
        'label_encoder': label_encoder,
        'inverse_mapping': inverse_mapping
    }

# ------------------------------------------------------------
# 5. DEPLOYMENT PIPELINE (WITH LABEL DECODING)
# ------------------------------------------------------------

def create_deployment_pipeline(training_results):
    """Create label-independent deployment inference."""
    kmeans = training_results['kmeans_model']
    scaler = training_results['scaler']
    classifiers = training_results['classifiers']
    used_features = training_results['used_fixed_features']
    class_features = training_results['classification_features']
    window_size = training_results.get('window_size', 100)
    stats = training_results.get('clustering_stats', ['mean', 'median', 'std', 'min', 'max'])
    inverse_mapping = training_results.get('inverse_mapping', {})
    states = [SlidingWindowStats(window_size) for _ in used_features]

    print(f"\nDeployment Ready: Window={window_size}, Stats={stats}, Rule=assigned cluster only")

    def predict(sample):
        cluster_vals = []
        for f_idx, feature in enumerate(used_features):
            states[f_idx].add(sample[feature])
            all_stats = states[f_idx].get_all_stats()
            cluster_vals.extend(all_stats[stat] for stat in stats)

        cluster_id = int(kmeans.predict(scaler.transform([cluster_vals]))[0])
        if cluster_id not in classifiers:
            raise RuntimeError(f"No trained classifier for cluster {cluster_id}")
        class_vals = np.array([sample.get(feature, 0.0) for feature in class_features]).reshape(1, -1)
        encoded = classifiers[cluster_id].predict(class_vals)[0]
        return {'cluster_id': cluster_id, 'prediction': inverse_mapping.get(encoded, encoded)}

    return predict

# ------------------------------------------------------------
# 6. FILE-BASED PIPELINE FUNCTIONS
# ------------------------------------------------------------

def run_pipeline_from_file(file_path, n_clusters=3, window_size=100,
                          clustering_stats=['mean', 'std', 'min', 'max'], **kwargs):
    """Run pipeline from CSV file"""
    df = pd.read_csv(file_path)
    if 'IdleTime' in df.columns:
        df = df.drop(columns=['IdleTime'])
    print(f"Loaded data from: {file_path}")
    print(f"Data shape: {df.shape}")
    
    return universal_clustering_pipeline(
        df=df, 
        n_clusters=n_clusters,
        window_size=window_size,
        clustering_stats=clustering_stats,
        **kwargs
    )

def run_pipeline_from_two_files(train_file, test_file, n_clusters=3, window_size=100,
                               clustering_stats=['mean', 'std', 'min', 'max'], **kwargs):
    """Run pipeline from separate train/test files"""
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    
    if 'IdleTime' in train_df.columns:
        train_df = train_df.drop(columns=['IdleTime'])
    if 'IdleTime' in test_df.columns:
        test_df = test_df.drop(columns=['IdleTime'])
    
    print(f"Train file: {train_file}, Shape: {train_df.shape}")
    print(f"Test file: {test_file}, Shape: {test_df.shape}")
    
    return universal_clustering_pipeline(
        df=train_df,
        external_test_df=test_df,
        n_clusters=n_clusters,
        window_size=window_size,
        clustering_stats=clustering_stats,
        **kwargs
    )


Testing with STRING labels...

CLUSTERING PIPELINE (Sliding Window) - Rule: Cluster Only (BATCH OPTIMIZED)
Stats: ['mean', 'std', 'min', 'max'], Window: 100, Features: SynAck, AckDat, TcpRtt

🔵 Encoding labels: ['HTTP' 'HTTPS' 'DNS']
Label mapping: {'DNS': np.int64(0), 'HTTP': np.int64(1), 'HTTPS': np.int64(2)}
Classes: 3

Data: 2000 samples, Classes: 3
  Class 0 (DNS): 582 (29.1%)
  Class 1 (HTTP): 1004 (50.2%)
  Class 2 (HTTPS): 414 (20.7%)

Train: 1599, Test: 401
Fixed features: ['SynAck', 'AckDat', 'TcpRtt'], Window: 100, Stats: ['mean', 'std', 'min', 'max']
Fixed features: ['SynAck', 'AckDat', 'TcpRtt'], Window: 100, Stats: ['mean', 'std', 'min', 'max']
Label coverage: target=1000, copied=1925

Classification features: 10 (excluding ['SynAck', 'AckDat', 'TcpRtt'])
Cluster 0: Trained on 262 samples, 3 classes
Cluster 1: Trained on 482 samples, 3 classes
Cluster 2: Trained on 855 samples, 3 classes

Evaluating test data (Rule: Cluster Only (BATCH OPTIMIZED))...

TEST RESULTS (Rule: 

In [ ]:
# scenario_1: run this cell independently
train_data = 'DATASETS/CDR-MLC/scale_1/Short/level_1.csv'
test_data = 'DATASETS/CDR-MLC/scale_1/Short/level_2.csv'

scenario_1_results = run_pipeline_from_two_files(
    train_file=train_data,
    test_file=test_data,
    n_clusters=3,
    window_size=3,
    clustering_stats=['mean', 'median', 'std', 'min', 'max']
)


In [ ]:
# scenario_2: run this cell independently
train_data = 'DATASETS/CDR-MLC/scale_1/Short/level_1.csv'
test_data = 'DATASETS/CDR-MLC/scale_1/Short/level_3.csv'

scenario_2_results = run_pipeline_from_two_files(
    train_file=train_data,
    test_file=test_data,
    n_clusters=3,
    window_size=3,
    clustering_stats=['mean', 'median', 'std', 'min', 'max']
)


In [ ]:
# scenario_3: run this cell independently
train_data = 'DATASETS/CDR-MLC/scale_1/Short/level_2.csv'
test_data = 'DATASETS/CDR-MLC/scale_1/Short/level_3.csv'

scenario_3_results = run_pipeline_from_two_files(
    train_file=train_data,
    test_file=test_data,
    n_clusters=3,
    window_size=3,
    clustering_stats=['mean', 'median', 'std', 'min', 'max']
)


In [ ]:
# scenario_4: run this cell independently
train_data = 'DATASETS/CDR-MLC/scale_1/Short/CDR-MLC-Shuffle.csv'
test_data = 'DATASETS/CDR-MLC/scale_1/Long/CDR-MLC-Shuffle.csv'

scenario_4_results = run_pipeline_from_two_files(
    train_file=train_data,
    test_file=test_data,
    n_clusters=3,
    window_size=3,
    clustering_stats=['mean', 'median', 'std', 'min', 'max']
)


In [ ]:
# scenario_5: run this cell independently
train_data = 'DATASETS/CDR-MLC/scale_1/Long/CDR-MLC-Shuffle.csv'
test_data = 'DATASETS/CDR-MLC/scale_1/Short/CDR-MLC-Shuffle.csv'

scenario_5_results = run_pipeline_from_two_files(
    train_file=train_data,
    test_file=test_data,
    n_clusters=3,
    window_size=3,
    clustering_stats=['mean', 'median', 'std', 'min', 'max']
)
